In [6]:
from __future__ import annotations

from dataclasses import dataclass
import numpy as np
import pandas as pd


MINUTES_IN_YEAR = 525_600
N30 = 30 * 24 * 60  # 43,200 minutes


def _require(cond: bool, msg: str) -> None:
    if not cond:
        raise ValueError(msg)


@dataclass(frozen=True)
class PreparedChain:
    """
    Clean boundary object: a single-expiration option chain prepared for VIX math.

    - Calls/Puts are indexed by strike with columns: bid, ask, mid.
    - strikes is the sorted union of strikes across both sides.
    """
    calls: pd.DataFrame
    puts: pd.DataFrame
    strikes: np.ndarray


def validate_chain_for_vix(chain: pd.DataFrame) -> None:
    """
    Optional validation for your upstream pipeline.

    This checks only "structural" assumptions so the calculation stays deterministic:
      - required columns exist
      - cp_flag is {C,P}
      - no duplicates per (cp_flag, strike)
      - finite/nonnegative quotes and bid <= ask
      - strikes finite and > 0

    The official methodology notes that series with null quotes or bid>ask are not candidates
    (and K0 null/bid>ask makes the index not calculable), so we treat those as invalid here. :contentReference[oaicite:1]{index=1}
    """
    required = {"strike", "cp_flag", "bid", "ask"}
    missing = required - set(chain.columns)
    _require(not missing, f"Missing required columns: {missing}")

    df = chain[list(required)].copy()
    df["cp_flag"] = df["cp_flag"].astype(str).str.upper()

    _require(df["cp_flag"].isin(["C", "P"]).all(), "cp_flag must be 'C' or 'P'.")
    _require(np.isfinite(df["strike"]).all() and (df["strike"] > 0).all(), "strike must be finite and > 0.")
    _require(np.isfinite(df["bid"]).all() and (df["bid"] >= 0).all(), "bid must be finite and >= 0.")
    _require(np.isfinite(df["ask"]).all() and (df["ask"] >= 0).all(), "ask must be finite and >= 0.")
    _require((df["bid"] <= df["ask"]).all(), "Found bid > ask (invalid quotes).")

    dup = df.duplicated(subset=["cp_flag", "strike"], keep=False)
    _require(not dup.any(), "Duplicate rows for (cp_flag, strike); deduplicate upstream.")


def prepare_chain_for_vix(chain: pd.DataFrame) -> PreparedChain:
    """
    Minimal preparation consistent with the official methodology:

      - Remove series with null quotes (explicit step in strike selection). :contentReference[oaicite:2]{index=2}
      - Remove series where bid > ask (officially not candidates for ATM and can make index incalculable). :contentReference[oaicite:3]{index=3}
      - Compute mid = (bid + ask)/2
      - Split into calls/puts indexed by strike (uniqueness enforced via verify_integrity=True)

    No "extra" sanity checks beyond what’s needed to make the downstream calculation deterministic.
    """
    cols = ["strike", "cp_flag", "bid", "ask"]
    missing = set(cols) - set(chain.columns)
    _require(not missing, f"Missing required columns: {missing}")

    df = chain.loc[:, cols].copy()
    df["cp_flag"] = df["cp_flag"].astype(str).str.upper()
    df = df[df["cp_flag"].isin(["C", "P"])]

    # Official step: remove null quotes. :contentReference[oaicite:4]{index=4}
    df = df.dropna(subset=["bid", "ask", "strike"])

    # Exclude invalid markets (bid > ask) per the official candidate rules. :contentReference[oaicite:5]{index=5}
    df = df[df["bid"] <= df["ask"]]

    df["mid"] = 0.5 * (df["bid"] + df["ask"])

    calls = (
        df[df["cp_flag"] == "C"]
        .set_index("strike", verify_integrity=True)[["bid", "ask", "mid"]]
        .sort_index()
    )
    puts = (
        df[df["cp_flag"] == "P"]
        .set_index("strike", verify_integrity=True)[["bid", "ask", "mid"]]
        .sort_index()
    )

    strikes = np.sort(df["strike"].unique())
    return PreparedChain(calls=calls, puts=puts, strikes=strikes)


@dataclass(frozen=True)
class TermResult:
    minutes_to_expiry: int
    T: float
    R: float
    K_atm: float
    F: float
    K0: float
    sigma2: float
    sum_piece: float
    qk: pd.DataFrame  # columns: strike, Q


def _select_K_atm_and_forward(prep: PreparedChain, R: float, T: float) -> tuple[float, float]:
    """
    Cboe step (Forward Price and K0):
      - K_atm is the strike minimizing |C_mid - P_mid| (tie -> lowest strike)
      - F = K + exp(RT) * (C_mid - P_mid)

    Note: the math methodology excludes series with null quotes or bid>ask from ATM candidacy;
    prepare_chain_for_vix() already filtered those. :contentReference[oaicite:6]{index=6}
    """
    common = prep.calls.join(prep.puts, how="inner", lsuffix="_C", rsuffix="_P")
    _require(len(common) > 0, "No strikes with both call and put quotes; cannot infer forward.")

    tmp = pd.DataFrame(
        {
            "strike": common.index.to_numpy(dtype=float),
            "abs_diff": (common["mid_C"] - common["mid_P"]).abs().to_numpy(),
        }
    ).sort_values(["abs_diff", "strike"], ascending=[True, True])

    K_atm = float(tmp.iloc[0]["strike"])
    row = common.loc[K_atm]
    F = float(K_atm + np.exp(R * T) * (row["mid_C"] - row["mid_P"]))
    return K_atm, F


def _strike_at_or_below(strikes: np.ndarray, x: float) -> float:
    leq = strikes[strikes <= x]
    _require(len(leq) > 0, "No strike <= forward; check strike grid / forward inference.")
    return float(leq[-1])


def _select_wing_two_consecutive_zero_bid_or_ask(wing: pd.DataFrame, *, direction: str) -> pd.DataFrame:
    """
    Cboe strike selection (current language):

      - Exclude any OTM option with bid == 0 OR ask == 0
      - Once two consecutive strikes are found with zero bid OR zero ask,
        exclude those observed options and consider no strikes beyond.

    This reflects the post–Feb 10, 2025 update in the official math methodology. :contentReference[oaicite:7]{index=7}
    """
    if wing.empty:
        return wing

    ordered = wing.sort_index(ascending=(direction == "up"))
    is_zero = (ordered["bid"].to_numpy() == 0) | (ordered["ask"].to_numpy() == 0)

    cut = None
    for i in range(len(is_zero) - 1):
        if is_zero[i] and is_zero[i + 1]:
            cut = i
            break

    if cut is None:
        # Keep all non-zero quotes
        return ordered.loc[~pd.Series(is_zero, index=ordered.index)]
    else:
        # Stop strictly before the first zero in the consecutive pair; also exclude any zero rows before the cut.
        kept = ordered.iloc[:cut]
        kept_zero = is_zero[:cut]
        return kept.loc[~pd.Series(kept_zero, index=kept.index)]


def _build_qk_table(prep: PreparedChain, K0: float) -> pd.DataFrame:
    """
    Cboe strike selection + pricing Q(K):

      - K < K0: OTM puts selected by the zero-quote rule
      - K > K0: OTM calls selected by the zero-quote rule
      - K = K0: include BOTH put and call; Q(K0) is their average midquote

    If all OTM puts or all OTM calls are excluded, the index cannot be calculated. :contentReference[oaicite:8]{index=8}
    """
    _require(K0 in prep.calls.index and K0 in prep.puts.index, "K0 must have both a call and a put quote.")

    puts_otm = prep.puts.loc[prep.puts.index < K0]
    calls_otm = prep.calls.loc[prep.calls.index > K0]

    puts_sel = _select_wing_two_consecutive_zero_bid_or_ask(puts_otm, direction="down")
    calls_sel = _select_wing_two_consecutive_zero_bid_or_ask(calls_otm, direction="up")

    _require(len(puts_sel) > 0, "All OTM puts excluded; volatility index cannot be calculated.")
    _require(len(calls_sel) > 0, "All OTM calls excluded; volatility index cannot be calculated.")

    q0 = float(0.5 * (prep.calls.loc[K0, "mid"] + prep.puts.loc[K0, "mid"]))

    qk = pd.concat(
        [
            pd.DataFrame({"strike": puts_sel.index.to_numpy(dtype=float), "Q": puts_sel["mid"].to_numpy(dtype=float)}),
            pd.DataFrame({"strike": np.array([K0], dtype=float), "Q": np.array([q0], dtype=float)}),
            pd.DataFrame({"strike": calls_sel.index.to_numpy(dtype=float), "Q": calls_sel["mid"].to_numpy(dtype=float)}),
        ],
        ignore_index=True,
    )

    qk = qk.groupby("strike", as_index=False)["Q"].mean().sort_values("strike").reset_index(drop=True)
    return qk


def _deltaK(strikes: np.ndarray) -> np.ndarray:
    """
    Cboe ΔK definition:
      - endpoints: adjacent difference
      - interior: (K_{i+1} - K_{i-1}) / 2
    """
    strikes = np.asarray(strikes, dtype=float)
    _require(len(strikes) >= 2, "Need at least 2 included strikes to compute ΔK.")

    dK = np.empty_like(strikes)
    dK[0] = strikes[1] - strikes[0]
    dK[-1] = strikes[-1] - strikes[-2]
    dK[1:-1] = 0.5 * (strikes[2:] - strikes[:-2])
    return dK


def compute_single_term_variance(prep: PreparedChain, *, R: float, minutes_to_expiry: int) -> TermResult:
    """
    Cboe single-term variance for one expiration.
    """
    _require(minutes_to_expiry > 0, "minutes_to_expiry must be positive.")
    T = minutes_to_expiry / MINUTES_IN_YEAR

    K_atm, F = _select_K_atm_and_forward(prep, R=R, T=T)
    K0 = _strike_at_or_below(prep.strikes, F)

    qk = _build_qk_table(prep, K0=K0)

    strikes = qk["strike"].to_numpy(dtype=float)
    Q = qk["Q"].to_numpy(dtype=float)
    dK = _deltaK(strikes)

    disc = np.exp(R * T)
    sum_piece = float(np.sum((dK / (strikes ** 2)) * disc * Q))

    # σ^2 = (2/T)*Σ[...] - (1/T)*(F/K0 - 1)^2
    sigma2 = float((2.0 / T) * sum_piece - (1.0 / T) * ((F / K0) - 1.0) ** 2)

    return TermResult(
        minutes_to_expiry=minutes_to_expiry,
        T=T,
        R=R,
        K_atm=K_atm,
        F=F,
        K0=K0,
        sigma2=sigma2,
        sum_piece=sum_piece,
        qk=qk,
    )

def interpolate_vix_30day(near: TermResult, next_: TermResult, target_minutes: int = N30) -> float:
    """
    Cboe constant-maturity interpolation to 30 days (in minutes), then VIX = 100*sqrt(var_30).
    """
    N1, N2 = near.minutes_to_expiry, next_.minutes_to_expiry
    _require(N1 < target_minutes < N2, "Need N1 < target_minutes < N2 for interpolation.")

    w1 = (N2 - target_minutes) / (N2 - N1)
    w2 = (target_minutes - N1) / (N2 - N1)

    var_30 = (w1 * near.T * near.sigma2 + w2 * next_.T * next_.sigma2) * (MINUTES_IN_YEAR / target_minutes)
    return float(100.0 * np.sqrt(var_30))

In [7]:
import numpy as np
import pandas as pd
import wrds
from zoneinfo import ZoneInfo

SPX_SECID = 108105
NY = ZoneInfo("America/New_York")

def _table_for_date(dt: pd.Timestamp) -> str:
    return f"optionm_all.opprcd{dt.year}"

def fetch_opprcd_slice(db: wrds.Connection, date: str,
                       exdate_min_days: int = 20,
                       exdate_max_days: int = 45) -> pd.DataFrame:
    d = pd.Timestamp(date)
    table = _table_for_date(d)
    ex_min = (d + pd.Timedelta(days=exdate_min_days)).date()
    ex_max = (d + pd.Timedelta(days=exdate_max_days)).date()

    sql = f"""
        SELECT
            secid, date, exdate, cp_flag, strike_price, best_bid, best_offer,
            am_settlement, ss_flag, contract_size, expiry_indicator
        FROM {table}
        WHERE secid = {SPX_SECID}
          AND date = '{d.date()}'
          AND exdate BETWEEN '{ex_min}' AND '{ex_max}'
          AND contract_size = 100
          AND ss_flag = '0'
          AND cp_flag IN ('C','P')
          AND best_bid IS NOT NULL
          AND best_offer IS NOT NULL
          AND strike_price IS NOT NULL
          AND best_bid <= best_offer
    """
    return db.raw_sql(sql)

def fetch_zero_curve(db: wrds.Connection, date: str) -> pd.DataFrame:
    d = pd.Timestamp(date).date()
    sql = f"""
        SELECT date, days, rate
        FROM optionm_all.zerocd
        WHERE date = '{d}'
        ORDER BY days
    """
    return db.raw_sql(sql)

def minutes_to_expiry(asof_date: str, exdate: pd.Timestamp, am_settlement: float) -> int:
    asof = pd.Timestamp(asof_date).tz_localize(NY) + pd.Timedelta(hours=16)
    exp = (
        pd.Timestamp(exdate).tz_localize(NY)
        + (pd.Timedelta(hours=9, minutes=30) if int(am_settlement) == 1 else pd.Timedelta(hours=16))
    )
    return int((exp - asof).total_seconds() // 60)

def pick_near_next_terms(df_day: pd.DataFrame, asof_date: str, target_minutes: int):
    df = df_day.copy()
    df["exdate"] = pd.to_datetime(df["exdate"])

    # VIX universe: AM-settled SPX + PM-settled SPXW end-of-week,
    # excluding PM expiries that coincide with AM expiries.
    am_expiries = set(df.loc[df["am_settlement"].fillna(0).astype(int) == 1, "exdate"].dt.date.unique())

    am = df["am_settlement"].fillna(0).astype(int)
    is_am = am == 1
    is_pm_eow = (am == 0) & (df["exdate"].dt.weekday == 4)  # PoC: Friday-only EOW
    is_same_date_pm = (am == 0) & (df["exdate"].dt.date.isin(am_expiries))

    df = df[(is_am | is_pm_eow) & (~is_same_date_pm)]

    series = df[["exdate", "am_settlement"]].drop_duplicates().copy()
    series["N"] = [
        minutes_to_expiry(asof_date, ex, am_)
        for ex, am_ in zip(series["exdate"], series["am_settlement"])
    ]
    series = series[series["N"] > 0].sort_values("N")

    near = series[series["N"] < target_minutes].tail(1)
    nxt  = series[series["N"] > target_minutes].head(1)
    if near.empty or nxt.empty:
        raise ValueError("Could not find Near/Next terms bracketing 30 days in the pulled expiry window.")

    e1 = (pd.Timestamp(near.iloc[0]["exdate"]), int(near.iloc[0]["am_settlement"]))
    e2 = (pd.Timestamp(nxt.iloc[0]["exdate"]), int(nxt.iloc[0]["am_settlement"]))
    return e1, e2

def interp_rate_from_zerocd(zc: pd.DataFrame, days_to_exp: float) -> float:
    """
    Linearly interpolate the OptionMetrics zero curve by days.
    Note: `rate` is continuously-compounded, but appears to be in *percent* units on WRDS for your sample.
    """
    x = zc["days"].to_numpy(dtype=float)
    y = zc["rate"].to_numpy(dtype=float)
    return float(np.interp(days_to_exp, x, y))

def rate_to_decimal(r: float) -> float:
    return r / 100.0 if r > 1.0 else r

def build_chain(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "strike": df["strike_price"].astype(float) / 1000.0,
        "cp_flag": df["cp_flag"].astype(str),
        "bid": df["best_bid"].astype(float),
        "ask": df["best_offer"].astype(float),
    })

In [ ]:

asof_date = "2025-03-10"
target_minutes = 30 * 24 * 60

db = wrds.Connection()

df = fetch_opprcd_slice(db, asof_date, exdate_min_days=20, exdate_max_days=45)
zc = fetch_zero_curve(db, asof_date)

(ex1, am1), (ex2, am2) = pick_near_next_terms(df, asof_date, target_minutes)

df1 = df[(pd.to_datetime(df["exdate"]) == ex1) & (df["am_settlement"].fillna(0).astype(int) == am1)]
df2 = df[(pd.to_datetime(df["exdate"]) == ex2) & (df["am_settlement"].fillna(0).astype(int) == am2)]

chain1 = build_chain(df1)
chain2 = build_chain(df2)

N1 = minutes_to_expiry(asof_date, ex1, am1)
N2 = minutes_to_expiry(asof_date, ex2, am2)

d = pd.Timestamp(asof_date)
days1 = N1 / (24*60)
days2 = N2 / (24*60)

R1 = rate_to_decimal(interp_rate_from_zerocd(zc, days1))
R2 = rate_to_decimal(interp_rate_from_zerocd(zc, days2))

prep1 = prepare_chain_for_vix(chain1)
prep2 = prepare_chain_for_vix(chain2)

near = compute_single_term_variance(prep1, R=R1, minutes_to_expiry=N1)
nxt  = compute_single_term_variance(prep2, R=R2, minutes_to_expiry=N2)

vix = interpolate_vix_30day(near, nxt, target_minutes=target_minutes)
print("VIX-like (EOD PoC):", vix)

Loading library list...
Done
VIX-like (EOD PoC): 28.058040751756224
